Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]

# Protocol 1 train: 3 augmentations from img1–3, 2 from img4
PROTOCOL1_TRAIN_INDICES = {
    1: [1, 2,],
    2: [1, 2],
    3: [1, 2],
    4: [1, 2]
}

# === PARAMETERS FOR (2D)^2PCA ===
NUM_COL_COMPONENTS = 47
NUM_ROW_COMPONENTS = 47

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    return cv2.fastNlMeansDenoising(image, h=h)

# === STRATEGY 2: LOAD EACH FINGER IMAGE AS SEPARATE SAMPLE ===
def load_fingers_strategy2_p1(subject_path, subject_id):
    finger_samples = []
    labels = []

    for img_num, aug_list in PROTOCOL1_TRAIN_INDICES.items():
        for aug_id in aug_list:
            for finger in FINGER_NUMS:
                fname = f"{subject_id}_{finger}_{img_num}_{aug_id}_Augmented.png"
                img_path = os.path.join(subject_path, fname)
                print(f"🖼️ Loading: {img_path}")

                if not os.path.exists(img_path):
                    print(f"❌ File not found: {img_path}")
                    continue

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, IMAGE_SIZE)
                img = img.astype(np.float32) / 255.0
                img_denoised = apply_denoising((img * 255).astype(np.uint8))
                img_denoised = img_denoised.astype(np.float32) / 255.0
                img_eq = exposure.equalize_hist(img_denoised)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                finger_samples.append(img_norm)
                label = f"{subject_id}_F{finger}_img{img_num}_aug{aug_id}"
                labels.append(label)
                print(f"✅ Added sample: {label}")

    return finger_samples, labels

# === LOAD TRAINING DATA ===
train_data = []
train_labels = []

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="📥 Loading Training Data - Protocol 1 Strategy 2"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n🔍 Processing subject: {subj}")
    samples, labels = load_fingers_strategy2_p1(subject_path, subj)
    train_data.extend(samples)
    train_labels.extend(labels)

train_data = np.array(train_data)
train_labels = np.array(train_labels)

# === (2D)^2PCA FUNCTIONS ===
def compute_2d2pca(images_2d, num_col_comp=NUM_COL_COMPONENTS, num_row_comp=NUM_ROW_COMPONENTS):
    print("\n⚙️ Computing (2D)^2PCA projection matrices...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n

    G_t = np.zeros((w, w))
    G_r = np.zeros((h, h))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        G_r += A @ A.T
        if i < 3:
            print(f"   ➕ Processed sample {i+1} for G_t and G_r")

    G_t /= n
    G_r /= n

    _, V = np.linalg.eigh(G_t)
    _, U = np.linalg.eigh(G_r)

    Wc = V[:, -num_col_comp:]
    Wr = U[:, -num_row_comp:]

    print(f"✅ Wc shape: {Wc.shape}, Wr shape: {Wr.shape}")
    return Wc, Wr

def project_2d2pca(images_2d, Wc, Wr):
    print("\n📐 Projecting images using (2D)^2PCA...")
    projected = []
    for i, img in enumerate(images_2d):
        feat = Wr.T @ img @ Wc  # Note: Corrected shape order
        projected.append(feat)
        if i < 3:
            print(f"   🧮 Projected shape of sample {i + 1}: {feat.shape}")
    return projected

def flatten_projected(projected_imgs):
    print("\n📦 Flattening projected features...")
    return np.array([img.flatten() for img in projected_imgs])

# === APPLY (2D)^2PCA ===
Wc, Wr = compute_2d2pca(train_data)
projected_features = project_2d2pca(train_data, Wc, Wr)
flat_features = flatten_projected(projected_features)

print(f"\n✅ Final Flattened Feature Shape: {flat_features.shape}")


Test

In [ ]:
# === TEST CONFIGURATION ===
TEST_BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
IMAGE_SIZE = (128, 60)

# Protocol 1 testing: use original images from img1–4 + 3rd augmented from img4
PROTOCOL1_TEST_FILES = [
    ("1", ""), ("2", ""), ("3", ""), ("4", ""),   # original image1–4
    ("1", "3_Augmented"),("2", "3_Augmented") ,("3", "3_Augmented") ,("4", "3_Augmented")     # 3rd augmented from image4
]

test_data = []
test_labels = []

# === FUNCTION: LOAD INDIVIDUAL FINGER IMAGES FOR TESTING (Strategy 2) ===
def load_fingers_test_strategy2_p1(subject_path, subject_id):
    finger_samples = []
    labels = []

    for img_num, suffix in PROTOCOL1_TEST_FILES:
        for finger in FINGER_NUMS:
            if suffix:
                fname = f"{subject_id}_{finger}_{img_num}_{suffix}.png"
                label_suffix = f"{img_num}_{suffix}"
            else:
                fname = f"{subject_id}_{finger}_{img_num}.png"
                label_suffix = f"{img_num}_orig"

            img_path = os.path.join(subject_path, fname)
            print(f"🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"❌ File not found: {img_path}")
                continue

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            img = img.astype(np.float32) / 255.0
            img_denoised = apply_denoising((img * 255).astype(np.uint8))
            img_denoised = img_denoised.astype(np.float32) / 255.0
            img_eq = exposure.equalize_hist(img_denoised)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            finger_samples.append(img_norm)
            label = f"{subject_id}_F{finger}_{label_suffix}"
            labels.append(label)
            print(f"✅ Test sample created: {label}")

    return finger_samples, labels

# === LOAD ALL TEST SUBJECTS ===
subject_dirs = sorted(os.listdir(TEST_BASE_PATH))
for subj in tqdm(subject_dirs, desc="🧪 Loading Test Data - Protocol 1 Strategy 2"):
    subject_path = os.path.join(TEST_BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n🔍 Processing subject: {subj}")
    samples, labels = load_fingers_test_strategy2_p1(subject_path, subj)
    test_data.extend(samples)
    test_labels.extend(labels)

test_data = np.array(test_data)
test_labels = np.array(test_labels)

print(f"\n📊 ✅ Final Test Data Shape: {test_data.shape}")
print(f"📌 First Few Test Labels: {test_labels[:5]}")

# === PROJECT TEST DATA USING (2D)^2PCA ===
proj_test_data = project_2d2pca(test_data, Wc, Wr)
flat_test_data = flatten_projected(proj_test_data)

print(f"✅ Final Flattened Test Feature Shape: {flat_test_data.shape}")


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(flat_test_data)

print("\n📤 Matching test samples using (2D)^2PCA features (Subject + Finger ID)...")

for i in range(total_tests):
    test_vector = flat_test_data[i]
    true_label = test_labels[i]  # e.g., "0001_F2_img1_aug1"

    # 📏 Compute Manhattan distances to all training vectors
    distances = np.sum(np.abs(flat_features - test_vector), axis=1)

    # 🏆 Nearest neighbor index
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "0001_F2_img2_aug3"

    # 🎯 Extract subject+finger IDs
    true_id_finger = "_".join(true_label.split("_")[:2])     # "0001_F2"
    pred_id_finger = "_".join(predicted_label.split("_")[:2])  # "0001_F2"

    # ✅ Match both subject and finger
    if pred_id_finger == true_id_finger:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:03d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Subject + Finger Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
